# ProtT5 ablation matrix and fair encoder comparison

**Status:** drafting code for review.

**When to run:** after notebook 01 establishes classical baselines and
notebook 02 fixes the evaluation protocols to be reported.

**Inputs:** complete frozen embedding caches and one selected split set.

**Outputs:** per-seed ablation metrics and predictions under
`tables/drafting_code` when enabled.

**Safety:** no test-driven model selection. Feature operator and common
dimension are specified before fitting. Pooling ablations require new
residue-level caches and are listed separately rather than faked from mean
pooled vectors. `QUICK_MODE=True` is a one-seed smoke run.

## 1. Setup and ablation registry

In [ ]:
from pathlib import Path
import os
import sys

# Keep the repository path separate from SNOOPPI_ROOT, which stores data/artifacts.
# In Colab the recommended clone location is /content/SNOOPPI.
DRAFT_DIR = Path.cwd()
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
    candidate = repository_root / "drafting_code"
    if candidate.exists():
        DRAFT_DIR = candidate
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    raise FileNotFoundError(
        "Cannot locate drafting_code/snooppi_utils.py. Open the notebook from the "
        "repository clone or set SNOOPPI_REPO_ROOT to the clone root."
    )
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL,
    SEQUENCE_A,
    SEQUENCE_B,
    canonical_pair_id,
    ensure_output_directories,
    experiment_paths,
    load_splits,
    normalize_sequence,
    sequence_id,
    split_manifest,
)

PATHS = experiment_paths()
ensure_output_directories(PATHS)
print("SNOOPPI root:", PATHS["root"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import GaussianRandomProjection, StandardScaler

from snooppi_utils import (
    build_pair_features,
    load_embedding_cache,
    select_threshold,
    summarize_predictions,
)

WRITE_OUTPUTS = False
RUN_TAG = "prott5_ablation_v1"
QUICK_MODE = True
SEEDS = [44] if QUICK_MODE else [41, 42, 43, 44, 45]

CACHE_PATHS = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}

PAIR_OPERATOR_ABLATIONS = {
    "sum": ("sum",),
    "absolute_difference": ("absdiff",),
    "product": ("product",),
    "sum_plus_absolute_difference": ("sum", "absdiff"),
    "sum_plus_product": ("sum", "product"),
    "absolute_difference_plus_product": ("absdiff", "product"),
    "full_symmetric": ("sum", "absdiff", "product"),
    "ordered_concat_diagnostic": ("concat",),
}
COMMON_DIMENSIONS = [256] if QUICK_MODE else [256, 512]

## 2. Load data and caches

In [ ]:
splits = load_splits(PATHS)
y_train = splits["train"][LABEL].to_numpy()
y_validation = splits["validation"][LABEL].to_numpy()
y_test = splits["test"][LABEL].to_numpy()

caches = {}
for encoder_name, cache_path in CACHE_PATHS.items():
    if cache_path.exists():
        caches[encoder_name], metadata = load_embedding_cache(cache_path)
        print(encoder_name, len(caches[encoder_name]), metadata)
    else:
        print("Skipping missing cache:", cache_path)
if "ProtT5" not in caches:
    raise FileNotFoundError("ProtT5 cache is required for the primary ablations")

## 3. Pair-operator ablations

These experiments isolate how two pooled protein embeddings are combined.
They do not answer whether ProtT5 pooling itself is superior. The ordered
concatenation diagnostic intentionally violates partner-swap invariance;
compare it with a swapped-partner test before considering it further.

In [ ]:
def fit_logistic(seed, x_train, x_validation, x_test):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=3_000,
            class_weight="balanced",
            random_state=seed,
        )),
    ])
    model.fit(x_train, y_train)
    validation_probability = model.predict_proba(x_validation)[:, 1]
    threshold = select_threshold(y_validation, validation_probability)
    test_probability = model.predict_proba(x_test)[:, 1]
    return threshold, test_probability


ablation_rows = []
prediction_rows = []
prot_cache = caches["ProtT5"]
for ablation_name, operations in PAIR_OPERATOR_ABLATIONS.items():
    x_train = build_pair_features(splits["train"], prot_cache, operations)
    x_validation = build_pair_features(splits["validation"], prot_cache, operations)
    x_test = build_pair_features(splits["test"], prot_cache, operations)
    for seed in SEEDS:
        threshold, probability = fit_logistic(seed, x_train, x_validation, x_test)
        metrics = summarize_predictions(y_test, probability, threshold)
        metrics.update({
            "experiment": "pair_operator",
            "encoder": "ProtT5",
            "ablation": ablation_name,
            "seed": seed,
            "feature_dimension": x_train.shape[1],
            "quick_mode": QUICK_MODE,
        })
        ablation_rows.append(metrics)
        prediction_rows.extend({
            "experiment": "pair_operator",
            "encoder": "ProtT5",
            "ablation": ablation_name,
            "seed": seed,
            "row_index": row_index,
            "label": int(label),
            "probability": float(score),
        } for row_index, (label, score) in enumerate(zip(y_test, probability)))

## 4. Common-dimension encoder comparison

Encoder vectors have different widths. Random projection to a common width
makes the downstream feature dimension identical while preserving a frozen,
seed-controlled transformation. This is one fairness check, not proof that
embedding width explains all performance differences.

In [ ]:
for common_dimension in COMMON_DIMENSIONS:
    for encoder_name, cache in caches.items():
        base_train = build_pair_features(splits["train"], cache)
        base_validation = build_pair_features(splits["validation"], cache)
        base_test = build_pair_features(splits["test"], cache)
        for seed in SEEDS:
            projector = GaussianRandomProjection(n_components=common_dimension, random_state=seed)
            projected_train = projector.fit_transform(base_train)
            projected_validation = projector.transform(base_validation)
            projected_test = projector.transform(base_test)
            threshold, probability = fit_logistic(seed, projected_train, projected_validation, projected_test)
            metrics = summarize_predictions(y_test, probability, threshold)
            ablation_name = f"random_projection_{common_dimension}"
            metrics.update({
                "experiment": "common_dimension",
                "encoder": encoder_name,
                "ablation": ablation_name,
                "seed": seed,
                "feature_dimension": common_dimension,
                "quick_mode": QUICK_MODE,
            })
            ablation_rows.append(metrics)
            prediction_rows.extend({
                "experiment": "common_dimension",
                "encoder": encoder_name,
                "ablation": ablation_name,
                "seed": seed,
                "row_index": row_index,
                "label": int(label),
                "probability": float(score),
            } for row_index, (label, score) in enumerate(zip(y_test, probability)))

ablation_metrics = pd.DataFrame(ablation_rows)
ablation_predictions = pd.DataFrame(prediction_rows)
summary = (
    ablation_metrics.groupby(["experiment", "encoder", "ablation"])
    .agg(
        macro_ap_mean=("macro_ap", "mean"),
        macro_ap_sd=("macro_ap", "std"),
        mcc_mean=("mcc", "mean"),
        brier_mean=("brier", "mean"),
        seeds=("seed", "nunique"),
    )
    .reset_index()
    .sort_values("macro_ap_mean", ascending=False)
)
display(summary.round(4))

## 5. Visual comparison and optional outputs

In [ ]:
pair_summary = summary.loc[summary["experiment"].eq("pair_operator")].sort_values("macro_ap_mean")
figure, axis = plt.subplots(figsize=(9, max(4.5, len(pair_summary) * 0.45)))
axis.barh(pair_summary["ablation"], pair_summary["macro_ap_mean"], xerr=pair_summary["macro_ap_sd"], color="#176B87")
axis.set(
    title="ProtT5 pair-operator ablations",
    xlabel="Test macro average precision (mean ± SD across seeds)",
    ylabel="",
    xlim=(0, 1),
)
figure.tight_layout()
plt.show()

if WRITE_OUTPUTS:
    output_files = {
        "metrics": PATHS["tables"] / f"{RUN_TAG}_metrics.csv",
        "predictions": PATHS["tables"] / f"{RUN_TAG}_test_predictions.csv",
        "summary": PATHS["tables"] / f"{RUN_TAG}_summary.csv",
        "figure": PATHS["figures"] / f"{RUN_TAG}_pair_operators.png",
    }
    existing = [path for path in output_files.values() if path.exists()]
    if existing:
        raise FileExistsError(f"Refusing to overwrite: {existing}")
    ablation_metrics.to_csv(output_files["metrics"], index=False)
    ablation_predictions.to_csv(output_files["predictions"], index=False)
    summary.to_csv(output_files["summary"], index=False)
    figure.savefig(output_files["figure"], dpi=300, bbox_inches="tight")
    print(output_files)
else:
    print("Preview only. Use a unique RUN_TAG before saving.")

## 6. What is still required to explain “ProtT5 is best”

| Hypothesis | Required controlled ablation |
|---|---|
| Embedding width | common-dimension projection/PCA, same head and split |
| Pair operator | the symmetric operator matrix above |
| Pooling | save residue embeddings; compare mean, max, attention, and CLS-like pooling |
| Preprocessing | harmonize normalization and truncation, then rebuild every cache |
| Hyperparameters | tune each encoder on validation with the same search budget |
| Random variation | repeated seeds plus paired bootstrap confidence intervals |
| Split leakage | repeat on cold-target/cold-family protocols |
| Pretraining/task match | report model data/provenance and test across species/families |

A statistically larger score on the current pair-row test split does not
establish encoder superiority on unseen targets or families.